<div style="text-align:center; padding:20px 0"><img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/></div>

# E-Commerce Analytics 360 — ShopAfrica+

## Notebook 4 — Dashboard Power BI & Storytelling

> Guide de création complet du dashboard Power BI **ShopAfrica+** (5 pages avec sidebar latérale DataProjectLab E-commerce). Conçu comme un script vidéo pas à pas avec captures d'écran intégrées.

| | |
|---|---|
| **Niveau** | Avancé |
| **Outils** | Power BI Desktop |
| **Durée estimée** | 6h à 8h |
| **Approche** | Mockup PowerPoint → conversion en PNG → backgrounds Power BI |

### Objectif business

Transformer les données SQL en un dashboard décisionnel **5 pages** permettant à M. Diallo de piloter : performance commerciale · produits · clients · funnel digital · satisfaction.

**Question finale à laquelle le dashboard doit répondre** :

> *"Que doit faire ShopAfrica+ dans les 3 prochains mois ?"*

---

## 1. Sources de données (7 fichiers)

### Fichiers à importer dans Power BI

| Fichier | Type | Rôle | Volume |
|---|---|---|---|
| `dim_customers.csv` | Dimension | Profil clients (segment, pays, ville) | 3 000 |
| `dim_products.csv` | Dimension | Catalogue produits (catégorie, prix) | 30 |
| `fact_orders.csv` | Fait | Commandes (date, client, canal, statut) | ~3 000 |
| `fact_order_items.csv` | Fait | Lignes de commande (produit, qté, CA, marge) | ~12 500 |
| `fact_reviews.csv` | Fait | Avis clients (rating, date, produit) | ~540 |
| `fact_web_logs.csv` | Fait | Comportement web (session, page, device) | ~1 100 |
| `clients_rfm_segments.csv` | Analytique | Segmentation RFM (6 segments) | 3 000 |

### URLs GitHub raw

```
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/ecommerce_analytics/corrige/outputs/dim_customers.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/ecommerce_analytics/corrige/outputs/dim_products.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/ecommerce_analytics/corrige/outputs/fact_orders.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/ecommerce_analytics/corrige/outputs/fact_order_items.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/ecommerce_analytics/corrige/outputs/fact_reviews.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/ecommerce_analytics/corrige/outputs/fact_web_logs.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/ecommerce_analytics/corrige/outputs/clients_rfm_segments.csv
```

### Procédure d'import

1. Power BI Desktop → **Obtenir des données → Web**
2. Coller chaque URL une par une
3. Cliquer **Transformer les données** pour vérifier les types en Power Query
4. **Accueil → Fermer & appliquer**

> ⚠️ **Important** : la table `fact_ecommerce_analytics` (issue de l'étape 3) n'est **pas importée** dans ce rapport. Les analyses passent directement par les tables de faits, ce qui simplifie le modèle et évite les doublons.

> 📸 **CAPTURE À INSÉRER** : `images/01_import_web_url.png` — Power BI Desktop → Obtenir des données → Web → coller l'URL

---

## 2. Désactiver Auto Date/Time (obligatoire)

**Avant toute manipulation du modèle** :

**Fichier → Options et paramètres → Options → Chargement des données (Fichier actuel)** → décocher **"Date/heure automatique pour le fichier actuel"**

Sans cette étape, Power BI crée 5 à 10 tables `LocalDateTable_*` parasites qui polluent le modèle, créent des relations fantômes et empêchent `SAMEPERIODLASTYEAR` / `PREVIOUSMONTH` de fonctionner correctement.

> 📸 **CAPTURE À INSÉRER** : `images/02_options_auto_datetime.png` — Options Power BI → Chargement des données → case Date/heure automatique décochée

---

## 3. Modèle de données — Schéma en étoile

### Architecture du modèle

```
                              Calendrier (dim temps)
                                     |
                                     v
           dim_customers --> fact_orders --> fact_order_items <-- dim_products
                  |                |                  |
                  v                v                  v
                  +---------> fact_reviews
                  |
                  +---------> fact_web_logs
                  |
                  +---------> clients_rfm_segments
```

### Relations à créer (10 relations, toutes Single)

| De | Colonne | Vers | Colonne | Cardinalité | Direction |
|---|---|---|---|---|---|
| `fact_orders` | `customer_id` | `dim_customers` | `customer_id` | N→1 | Single |
| `fact_reviews` | `customer_id` | `dim_customers` | `customer_id` | N→1 | Single |
| `fact_web_logs` | `customer_id` | `dim_customers` | `customer_id` | N→1 | Single |
| `clients_rfm_segments` | `customer_id` | `dim_customers` | `customer_id` | N→1 | Single |
| `fact_order_items` | `order_id` | `fact_orders` | `order_id` | N→1 | Single |
| `fact_order_items` | `product_id` | `dim_products` | `product_id` | N→1 | Single |
| `fact_reviews` | `order_id` | `fact_orders` | `order_id` | N→1 | Single |
| `fact_reviews` | `product_id` | `dim_products` | `product_id` | N→1 | Single |
| `fact_orders` | `order_date` | `Calendrier` | `Date` | N→1 | Single |
| `fact_web_logs` | `session_date` | `Calendrier` | `Date` | N→1 | Single |

### Points de vigilance

- **Direction Single** sur toutes les relations (jamais bidirectionnelle)
- Aucun chemin ambigu
- Si Power BI propose des relations auto-détectées superflues (ex: `fact_web_logs ↔ fact_orders` M:M), **les supprimer**
- `fact_web_logs[timestamp]` étant en DateTime, on relie via la colonne calculée `session_date` (voir [section 23](#col_calc))

> 📸 **CAPTURE À INSÉRER** : `images/03_modele_relations.png` — Affichage du modèle avec les 10 relations visibles

---

## 4. Table Calendrier

Modélisation → **Nouvelle table** → coller :

```dax
Calendrier =
ADDCOLUMNS(
    CALENDAR(DATE(2022,1,1), DATE(2024,12,31)),
    "Annee",        YEAR([Date]),
    "Mois_Num",     MONTH([Date]),
    "Mois_Nom",     FORMAT([Date], "MMMM", "fr-FR"),
    "Trimestre",    "T" & QUARTER([Date]),
    "Semaine",      WEEKNUM([Date]),
    "Jour_Semaine", FORMAT([Date], "dddd", "fr-FR"),
    "Est_Weekend",  IF(WEEKDAY([Date],2) >= 6, 1, 0),
    "Annee_Mois",   FORMAT([Date], "YYYY-MM")
)
```

**Marquer comme table de dates** : clic droit sur `Calendrier` dans le panneau Données → *Marquer comme table de dates* → choisir colonne `Date` → OK.

### Tri du mois nominal

Pour que Janvier apparaisse avant Février sur les axes :
1. Panneau Données → sélectionner colonne `Mois_Nom`
2. Onglet **Outils de colonne** → **Trier par colonne** → choisir `Mois_Num`

> 📸 **CAPTURE À INSÉRER** : `images/04_marquer_table_dates.png` — Boîte de dialogue "Marquer comme table de dates" avec colonne Date sélectionnée

---

## 5. Table _Mesures (placeholder)

Modélisation → **Nouvelle table** → coller :

```dax
_Mesures = {BLANK()}
```

Puis dans le panneau Données :
- Clic droit sur la colonne `Value` → **Masquer**
- La table `_Mesures` apparaît avec une icône calculatrice (réservée aux mesures)

> **Astuce** : le `_` au début du nom force Power BI à afficher cette table en premier dans la liste alphabétique.

> 📸 **CAPTURE À INSÉRER** : `images/05_table_mesures_value_masquee.png` — Table _Mesures avec colonne Value masquée

---

## 6. Design system — E-commerce

### Palette DataProjectLab E-commerce (navy / dark theme)

| Usage | Couleur | Hex |
|---|---|---|
| Fond principal | Navy foncé | `#1A1F2E` |
| Fond secondaire (cartes) | Navy moyen | `#232836` |
| Accent principal | **Bleu** | `#3B82F6` |
| Performance positive | **Vert teal** | `#10B981` |
| Accent secondaire | **Violet** | `#8B5CF6` |
| Attention | **Orange** | `#F59E0B` |
| Alerte | **Rouge** | `#EF4444` |
| Indigo (variations) | Indigo | `#6366F1` |
| Texte principal | Blanc | `#FFFFFF` |
| Texte secondaire | Gris clair | `#E8E8E8` |
| Texte discret | Gris moyen | `#888888` |

### Couleur dédiée par KPI Overview

| KPI | Couleur |
|---|---|
| CA | Bleu `#3B82F6` |
| Marge | Vert `#10B981` |
| Commandes | Teal `#14B8A6` |
| Clients | Violet `#8B5CF6` |
| Panier | Orange `#F59E0B` |
| Note | Rouge `#EF4444` |

### Palette segments RFM

| Segment | Couleur |
|---|---|
| Champions | `#10B981` |
| Fidèles | `#8B5CF6` |
| Gros dépensiers occasionnels | `#F59E0B` |
| Nouveaux prometteurs | `#3B82F6` |
| À réactiver | `#EF4444` |
| Dormants | `#888888` |

### Palette segments client

| Segment | Couleur |
|---|---|
| Premium | `#8B5CF6` |
| Standard | `#3B82F6` |
| Occasionnel | `#14B8A6` |
| Nouveau | `#F59E0B` |

### Typographie

| Usage | Police | Taille |
|---|---|---|
| Titres pages | Segoe UI bold | 22-26 px |
| Sous-titres | Segoe UI regular | 13-14 px |
| Valeurs KPI | Segoe UI bold | 34-42 px |
| Labels KPI | Segoe UI regular | 12-13 px |
| Pastilles variation | Segoe UI | 11 px |

---

## 7. Mockup PowerPoint et conversion en PNG

### Fichier de référence design

Le projet inclut **`mockup_ecommerce_powerbi.pptx`** : 5 slides reproduisant le dashboard final, plus un onglet bonus avec les mesures DAX.

Avant de pouvoir l'importer dans Power BI comme arrière-plan, il faut convertir chaque slide en PNG.

### Méthode 1 — PowerPoint natif (recommandée)

1. Ouvrir `mockup_ecommerce_powerbi.pptx` dans **PowerPoint**
2. **Fichier → Exporter → Modifier le type de fichier**
3. Sélectionner **PNG (Format Portable Network Graphics)** → cliquer **Enregistrer sous**
4. Choisir un dossier de sortie → **Enregistrer**
5. PowerPoint demande : *"Voulez-vous exporter toutes les diapositives ou uniquement la diapositive courante ?"* → cliquer **Toutes les diapositives**
6. PowerPoint crée un sous-dossier contenant `Slide1.png`, `Slide2.png`... `Slide5.png`

### Renommer les fichiers

Renommer manuellement OU lancer ce script PowerShell dans le dossier :

```powershell
1..5 | ForEach-Object {
    $num = "{0:D2}" -f $_
    Rename-Item -Path "Slide$_.PNG" -NewName "$num.png"
}
```

Résultat : `01.png`, `02.png`, `03.png`, `04.png`, `05.png`.

### Augmenter la résolution d'export (optionnel mais recommandé)

Par défaut, PowerPoint exporte à **96 DPI** (~960×540). Pour un rendu net en Power BI, monter à **150 DPI** (~1920×1080) :

1. Quitter PowerPoint complètement
2. **Win+R** → taper `regedit` → OK
3. Naviguer dans `HKEY_CURRENT_USER\Software\Microsoft\Office\16.0\PowerPoint\Options`
   *(remplacer 16.0 par ta version : 15.0 = 2013, 16.0 = 2016/2019/2021/365)*
4. Clic droit → **Nouveau → Valeur DWORD (32 bits)** → Nom : `ExportBitmapResolution`
5. Double-clic → **Base : Décimale** → **Données : 150** → OK
6. Relancer PowerPoint et exporter

### Méthode 2 — Convertisseur en ligne (alternative)

Si pas de PowerPoint :
1. Aller sur **cloudconvert.com** ou **convertio.co**
2. Uploader le PPTX
3. Sortie : **PNG**, qualité 150 DPI
4. Télécharger le ZIP, dézipper, renommer

> ⚠️ Pour des données client confidentielles, préférer la méthode locale (Méthode 1).

### Import des PNG comme arrière-plan dans Power BI

Pour chaque page Power BI :

1. Sélectionner la page (onglet en bas)
2. Cliquer dans une zone vide → volet **Format de la page** s'affiche
3. **Arrière-plan de page** → activer
4. **Image** → **Parcourir** → choisir le PNG correspondant
5. **Ajustement** : choisir **Ajuster**
6. **Transparence** : 0%

> 📸 **CAPTURE À INSÉRER** : `images/06_pptx_export_png.png` — PowerPoint Fichier → Exporter → PNG

> 📸 **CAPTURE À INSÉRER** : `images/07_powerbi_arriere_plan_image.png` — Power BI Format de la page → Arrière-plan → Image → Parcourir

---

## 8. Architecture de navigation — Sidebar latérale

### Sidebar gauche (présente sur les 5 pages)

Bande verticale à gauche (~200px de large), fond `#1A1F2E`, déjà incluse dans les PNG backgrounds.

**Contenu en haut** : logo DataProjectLab (carré bleu 40×40) + texte :
- `DataProjectLab` en blanc bold 14px
- `E-commerce` en gris `#888` 12px

### Menu de navigation (5 items avec icônes)

| Icône | Label | Page cible |
|---|---|---|
| 📊 | Overview | Page 1 |
| 🛒 | Produit | Page 2 |
| 👥 | Clients | Page 3 |
| 🔽 | Funnel digital | Page 4 |
| ⭐ | Satisfaction | Page 5 |

**Item actif** : fond navy clair `#232836`, texte blanc bold, icône bleue `#3B82F6`, bordure gauche 3px bleue (déjà dans le PNG par page).

### Implémentation des boutons (5 boutons par page)

Pour chaque item du menu :

1. **Insertion → Boutons → Vide**
2. Positionner exactement sur la zone de l'item dans le PNG
3. Format → **Action** → activer → **Type : Navigation de page** → cible = page correspondante
4. Format → **Texte du bouton** : désactivé
5. Format → **Forme** : remplissage transparent, bordure transparente

**Astuce** : sélectionner les 5 boutons sur la Page 1 (Ctrl+clic) → Ctrl+C → coller sur les 4 autres pages.

> 📸 **CAPTURE À INSÉRER** : `images/08_bouton_navigation_page.png` — Format du bouton → Action → Navigation de page → page cible

---

## 9. Slicers globaux

**Sur les 5 pages**, bandeau supérieur droit avec 2 groupes de slicers boutons :

### Slicer 1 — Année

| Propriété | Valeur |
|---|---|
| Champ | `Calendrier[Annee]` |
| Style | **Boutons arrondis** (single select) |
| Valeurs | `2022` · `2023` · `2024` |

### Slicer 2 — Trimestre

| Propriété | Valeur |
|---|---|
| Champ | `Calendrier[Trimestre]` |
| Style | **Boutons arrondis** (multi-select) |
| Valeurs | `T1` · `T2` · `T3` · `T4` |

### Style des boutons

- Forme arrondie (border-radius 20px)
- Fond transparent, bordure bleue `#3B82F6` 1px
- Bouton actif : fond bleu `#3B82F6`, texte blanc
- Bouton inactif : texte bleu `#3B82F6`, fond transparent

### Synchronisation

Affichage → **Synchroniser les segments** → cocher les 5 pages (Visible **et** Filtre).

---

## 10. Page 1 — Overview

**Background** : `01.png` (issu du mockup PPTX)

**Titre** : zone de texte → mesure `[Sous-titre Overview]` → retourne dynamiquement *"Performance globale · janv — déc 2024"* selon les filtres

### Ligne 1 : 6 KPI cards

Chaque card : visuel **Carte** posé sur la zone correspondante du PNG.

| # | Label | Mesure | Mesure variation | Couleur top |
|---|---|---|---|---|
| 1 | Chiffre d'affaires | `[CA Total]` (3,9 M€) | `[Variation CA %]` | 🔵 Bleu |
| 2 | Marge totale | `[Marge Totale]` (1,8 M€) + `[Taux de Marge]` 46,1% | `[Variation Marge %]` | 🟢 Vert |
| 3 | Commandes | `[Nb Commandes]` (3 018) | `[Variation Commandes %]` | 🟢 Teal |
| 4 | Clients actifs | `[Nb Clients]` (3 000) | `[Variation Clients %]` | 🟣 Violet |
| 5 | Panier moyen | `[Panier Moyen]` (1 297 €) | `[Variation Panier %]` | 🟠 Orange |
| 6 | Note moyenne | `[Note Moyenne]` (3,8 ★★★★☆) | `[Variation Note]` | 🔴 Rouge |

**Couleur de pastille variation** : Format → Couleur de la police → **fx** → Valeur du champ → mesure `[Couleur Variation X]` correspondante.

### Ligne 2 : Évolution CA + Donut catégories

**Gauche — Évolution CA 2024** (zone courbe)
- Visuel : **Graphique en aires**
- Axe X : `Calendrier[Mois_Nom]` (trié par `Mois_Num`)
- Axe Y : `[CA Total]`
- Couleur : bleu `#3B82F6` avec dégradé vers transparent

**Droite — CA par catégorie** (donut)
- Visuel : **Graphique en anneau**
- Légende : `dim_products[Categorie_Groupee]` *(colonne calculée Top 4 + Autres, voir [section 23](#col_calc))*
- Valeurs : `[CA Total]`
- Palette : Ordinateurs bleu, Smartphones teal, Autres gris, Tablettes violet, Audio orange

### Ligne 3 : CA par canal + Moyens de paiement

**Gauche — CA par canal de vente** (bar horizontal)
- Axe Y : `fact_orders[canal]`
- Axe X : `[CA Total]`
- Couleurs distinctes par canal

**Droite — Moyen de paiement** (donut)
- Légende : `fact_orders[moyen_paiement]`

> 📸 **CAPTURE À INSÉRER** : `images/09_page1_overview_final.png` — Aperçu de la Page 1 finalisée dans Power BI

---

## 11. Page 2 — Produit

**Background** : `02.png`

**Titre dynamique** : `[Sous-titre Produits]`

### Ligne 1 : 4 KPI cards

| # | Label | Mesure | Variation | Couleur |
|---|---|---|---|---|
| 1 | Qté vendue | `[Quantite Vendue]` (12 443) | `[Variation Quantite %]` | 🔵 Bleu |
| 2 | Revenu produits | `[CA Produit]` (4M€) | `[Variation CA %]` | 🔵 Bleu |
| 3 | Marge produits | `[Marge Produit]` (2M€) | `[Variation Marge %]` | 🟢 Vert |
| 4 | Produits actifs | `[Nb Produits Actifs]` (30) | `[Variation Produits Actifs %]` | 🟢 Vert |

### Ligne 2 : Top 10 produits + Scatter Revenu/Marge

**Gauche — Top 10 produits CA** (bar chart horizontal trié DESC)
- Axe Y : `dim_products[nom_produit]` (Top 10 filter)
- Axe X : `[CA Produit]`
- Couleurs dégradées bleu → violet → orange → rouge

**Droite — Revenu vs Marge par produit** (scatter bubble)
- Axe X : `[CA Produit]`
- Axe Y : `[Marge Produit]`
- Taille bulle : `[Quantite Vendue]`
- Détails : `dim_products[nom_produit]`

### Ligne 3 : Tableau détail produits

| Colonne | Champ | Format |
|---|---|---|
| Produit | `dim_products[nom_produit]` | Bold |
| Catégorie | `dim_products[categorie]` | Regular |
| Qté | `[Quantite Vendue]` | #,0 |
| Revenu | `[CA Produit]` | € #,0 |
| Marge | `[Marge Produit]` | € #,0 |
| % Marge | `[Taux de Marge Produit]` | 0,0% |

> 📸 **CAPTURE À INSÉRER** : `images/10_page2_produit_final.png` — Aperçu de la Page 2 finalisée

---

## 12. Page 3 — Clients

**Background** : `03.png`

**Titre dynamique** : `[Sous-titre Clients]`

### Ligne 1 : 3 KPI cards

| # | Label | Mesure | Variation |
|---|---|---|---|
| 1 | Clients actifs | `[Nb Clients]` (3 000) | `[Variation Clients %]` |
| 2 | Revenu moy / client | `[CA Moyen par Client]` (1 305 €) | `[Variation CA Moyen Client %]` |
| 3 | Commande moy / client | `[Nb Commandes par Client]` (1,0) | `[Variation Cmd Moyenne Client %]` |

### Ligne 2 : CA par segment client + Commandes par segment + Donut RFM

**Gauche — CA par segment client** (bar horizontal)
- Axe Y : `dim_customers[segment_client]`
- Axe X : `[CA Total]`
- **Couleur par formule** : Format → Couleurs des données → fx → `[Couleur Segment Client]`

**Milieu — Commandes par segment** (bar vertical)
- Axe X : `dim_customers[segment_client]`
- Axe Y : `[Nb Commandes]`
- Même couleur via `[Couleur Segment Client]`

**Droite — Répartition Clients RFM** (donut)
- Légende : `clients_rfm_segments[segment_rfm]`
- Valeurs : COUNT clients
- **Couleur par formule** : `[Couleur Segment RFM]`

### Ligne 3 : CA par segment RFM + Top clients

**Gauche — CA par segment RFM** (bar horizontal)
- Axe Y : `clients_rfm_segments[segment_rfm]`
- Axe X : `[CA Total]`
- Couleur via `[Couleur Segment RFM]`

**Droite — Top clients** (tableau)

| Colonne | Champ |
|---|---|
| ID Client | `dim_customers[customer_id]` |
| Segment | `dim_customers[segment_client]` (badge couleur via `Couleur Segment Client`) |
| Revenu | `[CA Total]` (trié DESC) |
| # Commande | `[Nb Commandes]` |

> 📸 **CAPTURE À INSÉRER** : `images/11_page3_clients_final.png` — Aperçu de la Page 3 finalisée

---

## 13. Page 4 — Funnel digital

**Background** : `04.png`

**Titre dynamique** : `[Sous-titre Funnel]`

### Ligne 1 : 4 KPI cards

| # | Label | Mesure | Variation |
|---|---|---|---|
| 1 | Vues totales | `[Nb Vues]` (1,1k) | `[Variation Vues %]` |
| 2 | Ajouts panier | `[Nb Ajouts Panier]` (320) | `[Variation Ajouts Panier %]` |
| 3 | Achats | `[Nb Achats]` (100) | `[Variation Achats %]` |
| 4 | Taux de conv. | `[Taux Conversion]` (9,4%) | `[Variation Taux Conversion]` (en pp) |

### Ligne 2 : Funnel d'achat + Conversions

**Gauche — Funnel d'achat** (View → Cart → Purchase)

Construction custom avec 4 zones empilées :

| Étape | Mesure | Couleur |
|---|---|---|
| Vues (1 059) | `[Nb Vues]` | bleu foncé |
| Ajout panier (320 — 30,2%) | `[Nb Ajouts Panier]` + `[Taux Panier sur Vues]` | vert |
| Checkout (149 — 46,6%) | `[Nb Checkout]` + `[Taux Checkout sur Panier]` | bleu clair |
| Achat confirmé (100 — 67,1%) | `[Nb Achats]` + `[Taux Achat sur Checkout]` | jaune |

**Encadré alerte en bas** : visuel **Carte** lié à `[Abandon Vues vers Panier]` qui retourne dynamiquement *"Perte majeure entre Vues et Panier — 69,8% d'abandons"*.

**Droite — Conversion par source** (bar horizontal)
- Axe Y : `fact_web_logs[source]`
- Axe X : `[Taux Conversion]`

**Droite (sous source) — Conversion par device** (bar horizontal)
- Axe Y : `fact_web_logs[device]`
- Axe X : `[Taux Conversion]`

### Ligne 3 : Tableau détail conversions

| Colonne | Champ |
|---|---|
| SOURCE | `fact_web_logs[source]` |
| DEVICE | `fact_web_logs[device]` |
| VUES | `[Nb Vues]` |
| PANIER | `[Nb Ajouts Panier]` |
| ACHATS | `[Nb Achats]` |
| TAUX CONVERSION | `[Taux Conversion]` |

> 📸 **CAPTURE À INSÉRER** : `images/12_page4_funnel_final.png` — Aperçu de la Page 4 finalisée

---

## 14. Page 5 — Satisfaction client

**Background** : `05.png`

**Titre dynamique** : `[Sous-titre Satisfaction]`

### Ligne 1 : 4 KPI cards

| # | Label | Mesure | Variation | Couleur |
|---|---|---|---|---|
| 1 | Note moyenne | `[Note Moyenne]` (3,8 ★) | `[Variation Note]` | 🔵 Bleu |
| 2 | Nombre d'avis | `[Nb Avis]` (538) | `[Variation Nb Avis %]` | 🔵 Bleu |
| 3 | Produits note <3 | `[Nb Produits Note <3]` (0) | `[Variation Produits Note <3]` | 🔴 Rouge |
| 4 | % Avis positifs | `[Pct Avis Positifs]` (69%) | `[Variation Pct Avis Positifs]` (pp) | 🟢 Vert |

### Ligne 2 : Distribution des notes + Produits faibles

**Gauche — Distribution des notes** (bar horizontal)
- Axe Y : `fact_reviews[rating]` (5, 4, 3, 2, 1)
- Axe X : `[Nb Avis]`
- Couleur conditionnelle : orange pour 3-5, rouge pour 1-2

**Gauche (dessous) — Évolution mensuelle de la note** (courbe)
- Axe X : `Calendrier[Mois_Nom]`
- Axe Y : `[Note Moyenne]`
- Couleur orange `#F59E0B`

**Droite — Produits avec notes les plus faibles** (bar horizontal)
- Axe Y : `dim_products[nom_produit]` (Bottom 4 filter)
- Axe X : `[Note Moyenne Produit]` (utilise `ALLEXCEPT` pour isoler par produit)

### Ligne 3 : Tableau détail satisfaction

| Colonne | Champ | Format |
|---|---|---|
| Produit | `dim_products[nom_produit]` | Bold |
| Catégorie | `dim_products[categorie]` | Regular |
| Note moy. | `[Note Moyenne Produit]` | ★ pastille colorée |
| Nb avis | `[Nb Avis]` | #,0 |
| Tendance | `[Tendance Note]` | → stable / ▲ +0,3 / ▼ -0,2 (couleur via `[Couleur Tendance Note]`) |

> 📸 **CAPTURE À INSÉRER** : `images/13_page5_satisfaction_final.png` — Aperçu de la Page 5 finalisée

---

## 15. Création des mesures DAX — Procédure

Toutes les mesures sont créées dans la table `_Mesures` et organisées par dossier d'affichage.

### Procédure de création d'une mesure

1. Panneau Données → clic droit sur `_Mesures` → **Nouvelle mesure**
2. Coller le code DAX dans la barre de formule
3. Onglet **Outils de mesure** :
   - **Format** : selon le type (% / nombre / décimal)
   - **Dossier d'affichage** : taper le nom du dossier (créé automatiquement à la première utilisation)

### Récapitulatif des dossiers (72 mesures)

| # | Dossier | Nb mesures | Rôle |
|---|---|---|---|
| 01 | `01 — KPIs de base` | 9 | CA, Marge, Cmd, Clients, Panier, Qté, Note, Avis |
| 02 | `02 — KPIs avancés` | 6 | Métriques dérivées + Tendance |
| 03 | `03 — Variations vs N-1` | 17 | ▲▼ vs année précédente avec UNICHAR |
| 04 | `04 — Couleurs Variation` | 18 | Hex pour formatage conditionnel |
| 05 | `05 — Couleurs Segments` | 2 | Palette dédiée RFM + Client |
| 06 | `06 — Funnel digital` | 10 | Vues / Panier / Checkout / Achats |
| 07 | `07 — Segment Produits` | 4 | KPIs au niveau produit |
| 08 | `08 — Sous-titres dynamiques` | 6 | Titres adaptés aux filtres |
|  | **Total** | **72** | |

> 📸 **CAPTURE À INSÉRER** : `images/14_nouvelle_mesure_dossier.png` — Création d'une mesure avec le dossier d'affichage rempli

---

## 16. Mesures — `01 — KPIs de base` (9 mesures)

```dax
CA Total = 
CALCULATE(
    SUM(fact_order_items[line_revenue]),
    fact_orders[order_status] = "Livree"
)

Marge Totale = 
CALCULATE(
    SUM(fact_order_items[line_margin]),
    fact_orders[order_status] = "Livree"
)

Taux de Marge = DIVIDE([Marge Totale], [CA Total], 0)

Nb Commandes = 
CALCULATE(
    DISTINCTCOUNT(fact_orders[order_id]),
    fact_orders[order_status] = "Livree"
)

Nb Clients = DISTINCTCOUNT(dim_customers[customer_id])

Panier Moyen = DIVIDE([CA Total], [Nb Commandes], 0)

Quantite Vendue = 
CALCULATE(
    SUM(fact_order_items[quantite]),
    fact_orders[order_status] = "Livree"
)

Note Moyenne = AVERAGE(fact_reviews[rating])

Nb Avis = COUNTROWS(fact_reviews)
```

**Format strings** : `CA Total`, `Marge Totale`, `Panier Moyen` → `#,0" €"` · `Taux de Marge` → `0.0%` · `Nb Commandes`, `Nb Clients`, `Quantite Vendue`, `Nb Avis` → `#,0` · `Note Moyenne` → `0.0`

---

## 17. Mesures — `02 — KPIs avancés` (6 mesures)

```dax
Nb Produits Actifs = 
CALCULATE(
    DISTINCTCOUNT(fact_order_items[product_id]),
    fact_order_items[quantite] > 0
)

CA Moyen par Client = DIVIDE([CA Total], [Nb Clients], 0)

Nb Commandes par Client = DIVIDE([Nb Commandes], [Nb Clients], 0)

Nb Produits Note <3 = 
VAR _result = 
    CALCULATE(
        DISTINCTCOUNT(fact_reviews[product_id]),
        FILTER(
            VALUES(fact_reviews[product_id]),
            CALCULATE(AVERAGE(fact_reviews[rating])) < 3
        )
    )
RETURN IF(ISBLANK(_result), 0, _result)

Pct Avis Positifs = 
DIVIDE(
    CALCULATE(COUNTROWS(fact_reviews), fact_reviews[rating] >= 4),
    [Nb Avis],
    0
)

Tendance Note = 
VAR _na = [Note Moyenne]
VAR _np = CALCULATE([Note Moyenne], DATEADD(Calendrier[Date], -1, MONTH))
VAR _delta = _na - _np
RETURN
    SWITCH(
        TRUE(),
        _delta > 0.1, UNICHAR(9650) & " +" & FORMAT(_delta, "0.0"),
        _delta < -0.1, UNICHAR(9660) & " " & FORMAT(_delta, "0.0"),
        UNICHAR(8594) & " stable"
    )
```

**Format strings** : `Nb Produits Actifs`, `Nb Produits Note <3` → `#,0` · `CA Moyen par Client` → `#,0" €"` · `Nb Commandes par Client` → `0.0` · `Pct Avis Positifs` → `0%`

---

## 18. Mesures — `03 — Variations vs N-1` (17 mesures)

### Pattern % standard (11 mesures)

```dax
Variation CA % = 
VAR _a = [CA Total]
VAR _p = CALCULATE([CA Total], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _pct = DIVIDE(_a - _p, _p, 0)
VAR _f = FORMAT(_pct, "0.0%;0.0%")
RETURN IF(_pct > 0, UNICHAR(9650) & " " & _f, UNICHAR(9660) & " " & _f)

Variation Marge %       = -- même pattern, remplacer [CA Total] par [Marge Totale]
Variation Commandes %   = -- même pattern, remplacer par [Nb Commandes]
Variation Clients %     = -- même pattern, remplacer par [Nb Clients]
Variation Panier %      = -- même pattern, remplacer par [Panier Moyen]
Variation Quantite %    = -- même pattern, remplacer par [Quantite Vendue]
Variation Produits Actifs % = -- remplacer par [Nb Produits Actifs]
Variation CA Moyen Client % = -- remplacer par [CA Moyen par Client]
Variation Cmd Moyenne Client % = -- remplacer par [Nb Commandes par Client]
Variation Vues %        = -- remplacer par [Nb Vues]
Variation Ajouts Panier % = -- remplacer par [Nb Ajouts Panier]
Variation Achats %      = -- remplacer par [Nb Achats]
Variation Nb Avis %     = -- remplacer par [Nb Avis]
```

### Pattern Note (delta absolu, 1 mesure)

```dax
Variation Note = 
VAR _a = [Note Moyenne]
VAR _p = CALCULATE([Note Moyenne], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _delta = _a - _p
RETURN IF(_delta > 0,
    UNICHAR(9650) & " +" & FORMAT(_delta, "0.0"),
    UNICHAR(9660) & " " & FORMAT(_delta, "0.0")
)
```

### Pattern points de pourcentage (pp, 2 mesures pour les taux)

```dax
Variation Taux Conversion = 
VAR _a = [Taux Conversion]
VAR _p = CALCULATE([Taux Conversion], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _delta = (_a - _p) * 100
VAR _f = FORMAT(_delta, "0.0""pp"";0.0""pp""")
RETURN IF(_delta > 0,
    UNICHAR(9650) & " +" & _f,
    UNICHAR(9660) & " " & _f
)

Variation Pct Avis Positifs = -- même pattern que Variation Taux Conversion
                                 -- remplacer [Taux Conversion] par [Pct Avis Positifs]
```

### Pattern delta absolu pour comptage (1 mesure)

```dax
Variation Produits Note <3 = 
VAR _a = [Nb Produits Note <3]
VAR _p = CALCULATE([Nb Produits Note <3], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _delta = _a - _p
RETURN
    SWITCH(
        TRUE(),
        _delta > 0, UNICHAR(9650) & " +" & FORMAT(_delta, "0"),
        _delta < 0, UNICHAR(9660) & " " & FORMAT(_delta, "0"),
        UNICHAR(8226) & " +0"
    )
```

---

## 19. Mesures — `04 — Couleurs Variation` (18 mesures)

À utiliser pour le **formatage conditionnel** des pastilles : Format → Couleur de la police → **fx** → **Mettre en forme par : Valeur du champ** → choisir la mesure couleur.

### Pattern standard (16 mesures)

```dax
Couleur Variation CA = 
VAR _p = DIVIDE(
    [CA Total] - CALCULATE([CA Total], SAMEPERIODLASTYEAR(Calendrier[Date])),
    CALCULATE([CA Total], SAMEPERIODLASTYEAR(Calendrier[Date])),
    0
)
RETURN SWITCH(TRUE(), _p > 0, "#10B981", _p < 0, "#EF4444", "#888888")
```

**Pattern à dupliquer pour** : `Couleur Variation Marge`, `Couleur Variation Commande`, `Couleur Variation Clients`, `Couleur Variation Panier moyen`, `Couleur Variation Note`, `Couleur Variation Quantite`, `Couleur Variation Produits Actifs`, `Couleur Variation CA Moyen Client`, `Couleur Variation Cmd Moyenne Client`, `Couleur Variation Vues`, `Couleur Variation Ajouts Panier`, `Couleur Variation Achats`, `Couleur Variation Nb Avis`, `Couleur Variation Pct Avis Positifs`.

### Pattern Taux Conversion (sémantique normale, 1 mesure)

```dax
Couleur Variation Taux Conversion = 
VAR _delta = [Taux Conversion] - CALCULATE([Taux Conversion], SAMEPERIODLASTYEAR(Calendrier[Date]))
RETURN SWITCH(TRUE(), _delta > 0, "#10B981", _delta < 0, "#EF4444", "#888888")
```

### Pattern Produits Note <3 (sémantique INVERSÉE, 1 mesure)

Plus de produits mal notés = MAUVAIS donc rouge.

```dax
Couleur Variation Produits Note <3 = 
VAR _delta = [Nb Produits Note <3] - CALCULATE([Nb Produits Note <3], SAMEPERIODLASTYEAR(Calendrier[Date]))
RETURN SWITCH(TRUE(), _delta > 0, "#EF4444", _delta < 0, "#10B981", "#888888")
```

### Couleur Tendance Note (basée sur DATEADD month, 1 mesure)

```dax
Couleur Tendance Note = 
VAR _na = [Note Moyenne]
VAR _np = CALCULATE([Note Moyenne], DATEADD(Calendrier[Date], -1, MONTH))
VAR _delta = _na - _np
RETURN SWITCH(TRUE(), _delta > 0.1, "#10B981", _delta < -0.1, "#EF4444", "#888888")
```

> 📸 **CAPTURE À INSÉRER** : `images/15_format_conditionnel_fx.png` — Format → Couleur → fx → Mettre en forme par : Valeur du champ

---

## 20. Mesures — `05 — Couleurs Segments` (2 mesures)

```dax
Couleur Segment RFM = 
SWITCH(
    SELECTEDVALUE(clients_rfm_segments[segment_rfm]),
    "Champions",                    "#10B981",
    "Fidèles",                      "#8B5CF6",
    "Gros dépensiers occasionnels", "#F59E0B",
    "Nouveaux prometteurs",         "#3B82F6",
    "À réactiver",                  "#EF4444",
    "Dormants",                     "#888888",
    "#888888"
)

Couleur Segment Client = 
SWITCH(
    SELECTEDVALUE(dim_customers[segment_client]),
    "Premium",     "#8B5CF6",
    "Standard",    "#3B82F6",
    "Occasionnel", "#14B8A6",
    "Nouveau",     "#F59E0B",
    "#888888"
)
```

---

## 21. Mesures — `06 — Funnel digital` (10 mesures)

```dax
Nb Sessions = DISTINCTCOUNT(fact_web_logs[session_id])

Nb Vues = 
CALCULATE(
    COUNTROWS(fact_web_logs),
    SEARCH("fiche_produit", fact_web_logs[page], 1, 0) > 0
)

Nb Ajouts Panier = 
CALCULATE(
    DISTINCTCOUNT(fact_web_logs[session_id]),
    SEARCH("panier", fact_web_logs[page], 1, 0) > 0
)

Nb Checkout = 
CALCULATE(
    DISTINCTCOUNT(fact_web_logs[session_id]),
    SEARCH("checkout", fact_web_logs[page], 1, 0) > 0
)

Nb Achats = 
CALCULATE(
    DISTINCTCOUNT(fact_web_logs[session_id]),
    SEARCH("confirmation", fact_web_logs[page], 1, 0) > 0
)

Taux Conversion = DIVIDE([Nb Achats], [Nb Vues], 0)

Taux Panier sur Vues = DIVIDE([Nb Ajouts Panier], [Nb Vues], 0)

Taux Checkout sur Panier = DIVIDE([Nb Checkout], [Nb Ajouts Panier], 0)

Taux Achat sur Checkout = DIVIDE([Nb Achats], [Nb Checkout], 0)

Abandon Vues vers Panier = 
VAR _ab = 1 - [Taux Panier sur Vues]
RETURN "Perte majeure entre Vues et Panier - " & FORMAT(_ab, "0.0%") & " d'abandons"
```

**Format strings** : mesures `Nb...` → `#,0` · mesures `Taux...` → `0.0%`

`Abandon Vues vers Panier` génère le texte de l'encadré rouge "Point de friction" en bas du visuel funnel.

---

## 22. Mesures — `07 — Segment Produits` (4 mesures)

```dax
CA Produit = 
CALCULATE(
    SUM(fact_order_items[line_revenue]),
    fact_orders[order_status] = "Livree"
)

Marge Produit = 
CALCULATE(
    SUM(fact_order_items[line_margin]),
    fact_orders[order_status] = "Livree"
)

Taux de Marge Produit = DIVIDE([Marge Produit], [CA Produit], 0)

Note Moyenne Produit = 
CALCULATE(
    AVERAGE(fact_reviews[rating]),
    ALLEXCEPT(fact_reviews, fact_reviews[product_id])
)
```

**Format strings** : `CA Produit`, `Marge Produit` → `#,0" €"` · `Taux de Marge Produit` → `0.0%` · `Note Moyenne Produit` → `0.0`

L'usage d'`ALLEXCEPT` sur `Note Moyenne Produit` permet d'afficher la note de chaque produit indépendamment du contexte de filtre (utile pour les visuels Top/Bottom).

---

## 23. Mesures — `08 — Sous-titres dynamiques` (6 mesures)

Ces mesures retournent un texte qui s'adapte aux filtres Année/Trimestre sélectionnés. À utiliser dans une zone de texte ou une **Carte** posée sous le titre de chaque page.

### Mesure helper `Periode Active`

```dax
Periode Active = 
VAR _ya = MIN(Calendrier[Annee])
VAR _yz = MAX(Calendrier[Annee])
VAR _ma = MIN(Calendrier[Mois_Num])
VAR _mz = MAX(Calendrier[Mois_Num])
VAR _maName = SWITCH(_ma,
    1, "janv", 2, "fev", 3, "mars", 4, "avr", 5, "mai", 6, "juin",
    7, "juil", 8, "aou", 9, "sept", 10, "oct", 11, "nov", 12, "dec")
VAR _mzName = SWITCH(_mz,
    1, "janv", 2, "fev", 3, "mars", 4, "avr", 5, "mai", 6, "juin",
    7, "juil", 8, "aou", 9, "sept", 10, "oct", 11, "nov", 12, "dec")
RETURN
    SWITCH(TRUE(),
        _ya <> _yz, _ya & " " & UNICHAR(8212) & " " & _yz,
        _ma = _mz, _maName & " " & _ya,
        _ma = 1 && _mz = 12, "janv " & UNICHAR(8212) & " dec " & _ya,
        _maName & " " & UNICHAR(8212) & " " & _mzName & " " & _ya
    )
```

### 5 sous-titres par page

```dax
Sous-titre Overview     = "Performance globale  ·  " & [Periode Active]
Sous-titre Produits     = "Performance & Rentabilite  ·  " & [Periode Active]
Sous-titre Clients      = "Segmentation & Valeur  ·  " & [Periode Active]
Sous-titre Funnel       = "Analyse du tunnel d'achat  ·  " & [Periode Active]
Sous-titre Satisfaction = "Avis & notes produits  ·  " & [Periode Active]
```

### Comportement attendu

| Filtres | `Periode Active` |
|---|---|
| Aucun filtre | `2022 — 2024` |
| Année = 2024 + tous T | `janv — dec 2024` |
| Année = 2024 + T2 et T3 | `avr — sept 2024` |
| Année = 2024 + T1 uniquement | `janv — mars 2024` |

---

## <a id='col_calc'></a>24. Colonnes calculées (2 colonnes)

### Colonne calculée `fact_web_logs[session_date]`

Nécessaire pour relier `fact_web_logs` à `Calendrier` (la colonne `timestamp` est en DateTime, pas en Date).

Panneau Données → clic droit sur `fact_web_logs` → **Nouvelle colonne** :

```dax
session_date = DATE(YEAR([timestamp]), MONTH([timestamp]), DAY([timestamp]))
```

Type : DateTime · Format : `yyyy-mm-dd` · cocher **Masquer** dans le panneau Champs.

Puis créer la relation `fact_web_logs[session_date] → Calendrier[Date]` (Many to One, Single).

### Colonne calculée `dim_products[Categorie_Groupee]`

Affiche les **Top 4 catégories par CA + "Autres"** sur le donut de la Page 1 (au lieu d'afficher les 14 catégories qui rendent le donut illisible).

Panneau Données → clic droit sur `dim_products` → **Nouvelle colonne** :

```dax
Categorie_Groupee = 
VAR _ca_current = 
    CALCULATE(
        SUM(fact_order_items[line_revenue]),
        fact_orders[order_status] = "Livree",
        ALLEXCEPT(dim_products, dim_products[categorie])
    )
VAR _allCats = 
    ADDCOLUMNS(
        ALL(dim_products[categorie]),
        "@ca",
        CALCULATE(
            SUM(fact_order_items[line_revenue]),
            fact_orders[order_status] = "Livree",
            ALLEXCEPT(dim_products, dim_products[categorie])
        )
    )
VAR _rank = 
    COUNTROWS(FILTER(_allCats, [@ca] > _ca_current)) + 1
RETURN
    IF(_rank <= 4, dim_products[categorie], "Autres")
```

> ⚠️ **Piège classique** : sans le `ALLEXCEPT(dim_products, dim_products[categorie])` dans le **CALCULATE intérieur** d'`_allCats`, le contexte de ligne du produit courant filtre `fact_order_items` au produit unique au lieu de la catégorie itérée. Le rang devient incorrect et tous les produits affichent leur catégorie au lieu de "Autres".

### Résultat attendu

Le donut de la Page 1 affiche désormais 5 segments :
- Ordinateurs (28%)
- Smartphones (27%)
- Tablettes (14%)
- Audio (8%)
- **Autres** (~22%, regroupant les 10 autres catégories)

> 📸 **CAPTURE À INSÉRER** : `images/16_colonne_calculee_dax.png` — Création d'une colonne calculée dans Power BI

---

## 25. Checklist de validation

### Import & modèle

- [ ] 7 fichiers CSV importés sans erreur
- [ ] Auto Date/Time désactivé
- [ ] Table `Calendrier` créée et marquée comme table de dates
- [ ] Table `_Mesures` créée (colonne Value masquée)
- [ ] 10 relations actives, aucun chemin ambigu
- [ ] Aucune table `LocalDateTable_*` parasite
- [ ] 5 backgrounds PNG importés sur les 5 pages (issus du PPTX)

### Colonnes calculées (2)

- [ ] `fact_web_logs[session_date]` (masquée)
- [ ] `dim_products[Categorie_Groupee]`

### Mesures DAX par dossier (72 mesures)

- [ ] **`01 — KPIs de base`** (9) : CA Total, Marge Totale, Taux de Marge, Nb Commandes, Nb Clients, Panier Moyen, Quantite Vendue, Note Moyenne, Nb Avis
- [ ] **`02 — KPIs avancés`** (6) : Nb Produits Actifs, CA Moyen par Client, Nb Commandes par Client, Nb Produits Note <3, Pct Avis Positifs, Tendance Note
- [ ] **`03 — Variations vs N-1`** (17) : 11× Variation X %, 1× Variation Note, 2× Variation pp (Taux Conversion + Pct Avis Positifs), 1× Variation Produits Note <3 (delta absolu), 2× autres patterns
- [ ] **`04 — Couleurs Variation`** (18) : 16× pattern standard, 1× Couleur Variation Taux Conversion, 1× Couleur Variation Produits Note <3 (inversé), Couleur Tendance Note
- [ ] **`05 — Couleurs Segments`** (2) : Couleur Segment RFM, Couleur Segment Client
- [ ] **`06 — Funnel digital`** (10) : Nb Sessions, Nb Vues, Nb Ajouts Panier, Nb Checkout, Nb Achats, Taux Conversion, Taux Panier sur Vues, Taux Checkout sur Panier, Taux Achat sur Checkout, Abandon Vues vers Panier
- [ ] **`07 — Segment Produits`** (4) : CA Produit, Marge Produit, Taux de Marge Produit, Note Moyenne Produit
- [ ] **`08 — Sous-titres dynamiques`** (6) : Periode Active, Sous-titre Overview, Sous-titre Produits, Sous-titre Clients, Sous-titre Funnel, Sous-titre Satisfaction

### Valeurs attendues sans filtre (année 2024)

| Mesure | Valeur |
|---|---|
| `[CA Total]` | 3,9 M€ |
| `[Marge Totale]` | 1,8 M€ |
| `[Taux de Marge]` | 46,1% |
| `[Nb Commandes]` | 3 018 |
| `[Nb Clients]` | 3 000 |
| `[Panier Moyen]` | 1 297 € |
| `[Note Moyenne]` | 3,8 |
| `[Quantite Vendue]` | 12 443 |
| `[Nb Produits Actifs]` | 30 |
| `[Nb Vues]` | 1 059 |
| `[Nb Ajouts Panier]` | 320 |
| `[Nb Achats]` | 100 |
| `[Taux Conversion]` | 9,4% |
| `[Nb Avis]` | 538 |
| `[Pct Avis Positifs]` | 69% |

### Pages

- [ ] Page 1 Overview : 6 KPI + variations + courbe CA + donut catégories + bar canal + donut paiement
- [ ] Page 2 Produit : 4 KPI + Top 10 produits + scatter + tableau
- [ ] Page 3 Clients : 3 KPI + 3 charts segments + CA RFM + Top clients
- [ ] Page 4 Funnel : 4 KPI + funnel custom + 2 conversions + tableau + alerte friction
- [ ] Page 5 Satisfaction : 4 KPI + distribution + évolution + Bottom 4 + tableau tendance

### Navigation & slicers

- [ ] Sidebar visible sur les 5 pages (via PNG arrière-plan)
- [ ] 5 boutons transparents par page (1 par item de menu) avec Action Navigation de page
- [ ] Slicer Année (boutons 2022/2023/2024) en haut à droite
- [ ] Slicer Trimestre (boutons T1/T2/T3/T4) en haut à droite
- [ ] Slicers synchronisés sur les 5 pages

---

## 26. Storytelling — Présentation au comité de direction (5 min)

### Séquence narrative pour M. Diallo

**1. Performance globale (Page Overview) — 1 min**

> *"Voici où on en est : CA 3,9M€ (-54% vs 2023), marge 1,8M€, 3 018 commandes. Le CSAT se maintient à 3,8/5 et 69% d'avis positifs."*

**2. Moteurs de performance (Page Produit) — 1 min**

> *"iPhone 15 Pro et MacBook Air M3 concentrent 25% du CA. 30 produits actifs portent toute la performance — risque de concentration extrême."*

**3. Valeur client (Page Clients) — 1 min**

> *"3 000 clients actifs, revenu moyen 1 305 € (-54%). Segments RFM : 43% Dormants + À réactiver — la priorité absolue est la réactivation de cette base."*

**4. Conversion digitale (Page Funnel) — 1 min**

> *"Taux conversion 9,4% (+1,7pp vs 2023), donc le funnel s'améliore. MAIS point de friction majeur : 69,8% d'abandons entre Vues et Panier — chantier prioritaire."*

**5. Qualité perçue (Page Satisfaction) — 1 min**

> *"Note 3,8/5, 69% avis positifs. 4 produits sous la barre des 3,5 étoiles à surveiller : Dell Inspiron 15, Samsung Tab S9, iPad Air, Sony WH-1000XM5."*

### 3 recommandations chiffrées

1. **Stopper l'hémorragie CA (-54%)** — investigation urgente sur Ordinateurs + Smartphones
2. **Reconquête clients dormants (43% base)** — campagne email personnalisée fin Q1
3. **Fix du tunnel d'achat (-69,8% Vues→Panier)** — rework fiche produit + bouton panier

**Objectif 90 jours : ramener le CA en croissance et stabiliser la qualité produit.**

---

> L'apprenant doit pouvoir répondre à la question :
> **"Que doit faire ShopAfrica+ dans les 3 prochains mois ?"**
> La réponse est contenue dans les 5 pages du dashboard — pas ailleurs.

---

## 27. Annexes

### Annexe A — Règles techniques DAX non négociables

- `FILTER(table, [Mesure] = "valeur")` plutôt que `mesure = "valeur"` directement dans `CALCULATE`
- `VAR ... RETURN` pour toutes les mesures avec logique conditionnelle
- `DIVIDE()` pour toutes les divisions (gestion des zéros)
- `IF(ISBLANK(_result), 0, _result)` sur les mesures de comptage qui peuvent être vides (cas `Nb Produits Note <3`)
- `SAMEPERIODLASTYEAR(Calendrier[Date])` nécessite la table Calendrier marquée comme table de dates
- `RANKX(ALL(table[col]), ...)` : `ALL` est obligatoire pour ranker sur tous les éléments
- `SEARCH("texte", colonne, 1, 0) > 0` pour matcher un texte partiel sans crash sur BLANK
- Pour les flèches : `UNICHAR(9650)` = ▲, `UNICHAR(9660)` = ▼, `UNICHAR(8594)` = →, `UNICHAR(8226)` = •, `UNICHAR(8212)` = —

### Annexe B — Pièges classiques rencontrés sur ce projet

| Piège | Symptôme | Solution |
|---|---|---|
| Auto Date/Time actif | Tables `LocalDateTable_*` parasites | Désactiver dans Options *avant* la modélisation |
| `fact_web_logs[timestamp]` en DateTime | Relation impossible avec `Calendrier[Date]` | Créer colonne calculée `session_date` (DATE only) |
| Mesure de couleur dans la légende d'un scatter | "Ce champ ne peut pas être utilisé ici" | Convertir en colonne calculée sur la table de détails |
| `Categorie_Groupee` retourne tous les noms (pas "Autres") | Tous les produits affichent leur catégorie | Ajouter `ALLEXCEPT` dans le CALCULATE intérieur |
| `Progression Label` retourne juste "%" | `SELECTEDVALUE` retourne BLANK sur plusieurs valeurs | Utiliser `MAX()` au lieu de `SELECTEDVALUE` |
| Variation "vs 2023" hardcodé | Suffixe ne suit pas l'année sélectionnée | `MAX(Calendrier[Annee]) - 1` pour suffixe dynamique |
| Mois non triés sur axe X | Janvier après Décembre alphabétique | Outils de colonne → Trier par colonne → Mois_Num |

### Annexe C — Liste des captures à insérer

Voici la liste consolidée des captures à intégrer dans le notebook (à placer dans `images/`) :

| # | Fichier | Étape | Section |
|---|---|---|---|
| 01 | `01_import_web_url.png` | Power BI → Obtenir des données → Web | §1 |
| 02 | `02_options_auto_datetime.png` | Désactivation Auto Date/Time | §2 |
| 03 | `03_modele_relations.png` | Vue du modèle avec 10 relations | §3 |
| 04 | `04_marquer_table_dates.png` | Marquer Calendrier comme table de dates | §4 |
| 05 | `05_table_mesures_value_masquee.png` | Table _Mesures avec colonne Value masquée | §5 |
| 06 | `06_pptx_export_png.png` | PowerPoint → Exporter → PNG | §7 |
| 07 | `07_powerbi_arriere_plan_image.png` | Format de la page → Arrière-plan → Image | §7 |
| 08 | `08_bouton_navigation_page.png` | Bouton transparent → Action → Navigation | §8 |
| 09 | `09_page1_overview_final.png` | Aperçu Page 1 finalisée | §10 |
| 10 | `10_page2_produit_final.png` | Aperçu Page 2 finalisée | §11 |
| 11 | `11_page3_clients_final.png` | Aperçu Page 3 finalisée | §12 |
| 12 | `12_page4_funnel_final.png` | Aperçu Page 4 finalisée | §13 |
| 13 | `13_page5_satisfaction_final.png` | Aperçu Page 5 finalisée | §14 |
| 14 | `14_nouvelle_mesure_dossier.png` | Création mesure avec dossier d'affichage | §15 |
| 15 | `15_format_conditionnel_fx.png` | Format → Couleur → fx → Valeur du champ | §19 |
| 16 | `16_colonne_calculee_dax.png` | Création colonne calculée | §24 |

**Total : 16 captures.** Stocker dans `D:\DataProjectLab\DataProjectLab-projects\projets\ecommerce_analytics\powerbi\images\`.

### Annexe D — Mapping rapide visuel ↔ mesures

| Visuel dashboard | Mesures utilisées |
|---|---|
| Card CA + pastille | `CA Total` + `Variation CA %` + `Couleur Variation CA` |
| Card Marge + pastille | `Marge Totale` + `Variation Marge %` + `Couleur Variation Marge` |
| Card Commandes + pastille | `Nb Commandes` + `Variation Commandes %` + `Couleur Variation Commande` |
| Card Clients + pastille | `Nb Clients` + `Variation Clients %` + `Couleur Variation Clients` |
| Card Panier + pastille | `Panier Moyen` + `Variation Panier %` + `Couleur Variation Panier moyen` |
| Card Note + pastille | `Note Moyenne` + `Variation Note` + `Couleur Variation Note` |
| Sous-titre Overview | `Sous-titre Overview` (utilise `Periode Active`) |
| Donut catégories | `dim_products[Categorie_Groupee]` (colonne calculée) + `CA Total` |
| Funnel — étapes | `Nb Vues`, `Nb Ajouts Panier` + `Taux Panier sur Vues`, `Nb Checkout` + `Taux Checkout sur Panier`, `Nb Achats` + `Taux Achat sur Checkout` |
| Funnel — alerte friction | `Abandon Vues vers Panier` |
| Donut RFM | `clients_rfm_segments[segment_rfm]` + `Couleur Segment RFM` |
| Bar CA segment client | `dim_customers[segment_client]` + `CA Total` + `Couleur Segment Client` |
| Bottom 4 produits | `dim_products[nom_produit]` + `Note Moyenne Produit` |
| Tableau Tendance Note | `Tendance Note` + `Couleur Tendance Note` |

### Annexe E — Fichiers livrables

| Fichier | Localisation | Usage |
|---|---|---|
| `E-commerce Dahboard.pbix` | `powerbi/` | Rapport Power BI final |
| `mockup_ecommerce_powerbi.pptx` | `powerbi/` | Maquette statique 5 slides — à convertir en PNG |
| `01.png` à `05.png` | `powerbi/` (générés via export PowerPoint) | Backgrounds à importer dans Power BI |
| `ressources_powerbi.ipynb` | `powerbi/` | Ce document — guide de création |
| `images/01-16.png` | `powerbi/images/` | Captures du tutoriel |

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.